# RSA Key Generation

RSA is built entirely on number theory that was developed over centuries, long before anyone imagined it would be used in cryptography.

## The Setup

Key generation in RSA has four steps:

1. Pick two large primes $p$ and $q$. Compute $n = pq$.
2. Compute $\phi(n) = (p-1)(q-1)$
3. Pick a public exponent $e$ such that $\gcd(e, \phi(n)) = 1$.
4. Compute the private exponent $d = e^{-1} \pmod{\phi{n}}$.

Your public key is $(e, n)$. Your private key is $(d, n)$. That's it.

## Step 1: Choose $p$, $q$, and $n$

In practice, $p$ and $q$ are randomly generated primes of equal bit length.

Typically they are $1024$ bits each, giving a $2048$-bit modulus.

I'll use small primes here so that the arithmetic is a bit more tangible.

In [1]:
using Primes

p = 61
q = 53
n = p * q

println("p = $p")
println("q = $q")
println("n = p * q = $n")

p = 61
q = 53
n = p * q = 3233


## Step 2: Compute $\phi(n)$

$\phi(n)$ counts how many integers in $1$ to $n$ are coprime to $n$. For $n = pq$, the formula is:

$$\phi(pq) = (p-1)(q-1)$$

>See the Fermat and Euler notebook in Number Theory for the derivation.

In [2]:
phi_n = (p - 1) * (q - 1)

println("phi(n) = $(phi_n)")
println("totient check: $(totient(n))")

phi(n) = 3120
totient check: 3120


## Step 3: Choose the Public Exponent $e$

$e$ must satisfy $\gcd(e, \phi(n)) = 1$. In real RSA, $e = 65537$ is almost universally used. It's prime, coprime to virtually any $\phi(n)$, and has a sparse binary representation that makes modular exponentiation fast.

For our stripped-down version, we can find the first valid $e$ automatically:

In [3]:
function find_e(phi_n)
    i = 1
    while true
        if gcd(prime(i), phi_n) == 1
            return prime(i)
        end
        i += 1
    end
end

e = find_e(phi_n)
println("e = $e")
println("gcd(e, phi_n) = $(gcd(e, phi_n))")

e = 7
gcd(e, phi_n) = 1


## Step 4: Compute the Private Exponent $d$

$d$ is the modular inverse of $e \text{ mod } \phi(n)$. It satisfies:

$$ed \equiv 1 \pmod{\phi(n)}$$

>This is computed using the Extended Euclidean Algorithm. See the Modular Inverses notebook in Number Theory for the derivation.

In [5]:
d = invmod(e, phi_n)

println("d = $d")
println("Verify: (e * d) % phi_n = $((e * d) % phi_n)") # should be 1

d = 1783
Verify: (e * d) % phi_n = 1


## The Keys

In [6]:
println("Public key:  (e = $e, n = $n)")
println("Private key: (d = $d, n = $n)")

Public key:  (e = 7, n = 3233)
Private key: (d = 1783, n = 3233)


$n$ appears in both keys since it's the modulus for all the arithmetic and is public. What's secret is $d$.

An attacker who knows $n$ and $e$ would need to factor $n$ back into $p$ and $q$ to compute $\phi(n)$ and recover $d$. For a $2048$-bit $n$, this is infeasible with any known algorithm.

## Putting it all together

In [7]:
function keygen(p, q)
    n = p * q
    phi_n = (p - 1) * (q - 1)
    e = find_e(phi_n)
    d = invmod(e, phi_n)
    return (e, n), (d, n) # public and private keys
end

public_key, private_key = keygen(p, q)
println("Public key:  $public_key")
println("Private key: $private_key")

Public key:  (7, 3233)
Private key: (1783, 3233)


## With Cryptographic-Scale Primes

The same code works, just use larger primes.

In [12]:
using Random

p_big = nextprime(rand(big(2)^511 : big(2)^512))
q_big = nextprime(rand(big(2)^511 : big(2)^512)) # two random 512-bit primes

public_key_big, private_key_big = keygen(p_big, q_big)

println("n ($(ndigits(public_key_big[2])) digits)")
println("e = $(public_key_big[1])")
println("d = $(private_key_big[1])")

n (309 digits)
e = 5
d = 43034733475038481651421464939273201038314030308302284655568068801156513150076093087228621301707147665224755018848217941107166741885691832876534495469389224819198717313876226485172281174163862399293102041162874898545837939482024928524107588150710559701965299167199194578986960838384426583044687781842854987469


Encryption and decryption are covered in the next notebook.